# Statistical Analysis Shortlist
**Goal**: Reduce 72 results to most promising configurations

## Imports

In [13]:
import sys
import math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import t

### Setup

In [14]:
PRIMARY_METRIC = "f1_macro_mean"   # primary metric
PRIMARY_STD    = "f1_macro_std"
N_COL          = "num_folds"
ALPHA          = 0.05  # 95% CI
TOP_N = 10 
OUT_DIR = Path("./results")
RESULTS_CSV = "adult_results.csv"
RERUN_CSV = "shortlist_for_rerun_adult.csv"

## Load Results CSV

In [15]:
csv_path = OUT_DIR / RESULTS_CSV

if not csv_path.exists():
    raise SystemExit(f"CSV not found: {csv_path}")

df = pd.read_csv(csv_path)

# csv format check
needed_columns = [PRIMARY_METRIC, PRIMARY_STD, N_COL,
        "dataset","metric","k","vote","retention"]
missing = [c for c in needed_columns if c not in df.columns]
if missing:
    raise SystemExit(f"Missing column(s): {missing}")


## Confidence Interval

In [16]:
def mean_ci(mean: float, sd: float, n: int, alpha: float = 0.05):
    """
    (1-alpha) t-based CI for a mean across n folds:
      mean ± t_{1-alpha/2, n-1} * sd / sqrt(n)
    """
    if n is None or n < 2 or pd.isna(sd):
        return (np.nan, np.nan)
    tcrit = t.ppf(1 - alpha/2.0, df=n-1)
    half = tcrit * (sd / math.sqrt(n))
    return (mean - half, mean + half)

def ci_overlap(a, b) -> bool:
    """True if intervals [a1, a2] and [b1, b2] overlap."""
    a1, a2 = a
    b1, b2 = b
    if any(pd.isna(x) for x in (a1, a2, b1, b2)):
        return False
    return not (a2 < b1 or b2 < a1)

## Keeping all results statistically the same at 95% CI

In [17]:
# compute 95% CIs for the primary metric
cis = df.apply(
    lambda r: mean_ci(r[PRIMARY_METRIC], r[PRIMARY_STD], int(r[N_COL]), ALPHA),
    axis=1
)
df[f"{PRIMARY_METRIC}_ci_lo"], df[f"{PRIMARY_METRIC}_ci_hi"] = zip(*cis)

# find the best by mean of primary metric
best_idx = df[PRIMARY_METRIC].idxmax()
best_row = df.loc[best_idx]
best_ci = (best_row[f"{PRIMARY_METRIC}_ci_lo"], best_row[f"{PRIMARY_METRIC}_ci_hi"])

# keep rows whose CI overlaps best CI
survivors = df[df.apply(
    lambda r: ci_overlap(
        (r[f"{PRIMARY_METRIC}_ci_lo"], r[f"{PRIMARY_METRIC}_ci_hi"]),
        best_ci
    ), axis=1
)].copy()

# sort for viewing: higher mean, then lower std
sort_cols = [PRIMARY_METRIC, PRIMARY_STD]
sort_asc  = [False, True]

survivors = survivors.sort_values(sort_cols, ascending=sort_asc)
survivors

,dataset,metric,k,vote,retention,num_folds,n_train_mean,n_test_mean,fit_time_s_mean,fit_time_s_std,...,f1_macro_std,precision_weighted_mean,precision_weighted_std,recall_weighted_mean,recall_weighted_std,f1_weighted_mean,f1_weighted_std,confusion_matrix_json,f1_macro_mean_ci_lo,f1_macro_mean_ci_hi
41,adult,cosine,7,borda,RetentionPolicy.ALWAYS_RETAIN,10,43957.8,4884.2,0.109486,0.004580,...,0.007056,0.776686,0.004927,0.791184,0.004314,0.780747,0.004624,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3345...",0.680923,0.691018
68,adult,cosine,7,borda,RetentionPolicy.DD_RETENTION,10,43957.8,4884.2,0.062214,0.001742,...,0.007323,0.776672,0.005157,0.791184,0.004551,0.780729,0.004841,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3345...",0.680687,0.691165
59,adult,cosine,7,borda,RetentionPolicy.DIFFERENT_CLASS_RETENTION,10,43957.8,4884.2,0.065031,0.001216,...,0.007338,0.776285,0.005212,0.790611,0.004631,0.780407,0.004900,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3341...",0.680515,0.691013
40,adult,cosine,5,borda,RetentionPolicy.ALWAYS_RETAIN,10,43957.8,4884.2,0.108597,0.009831,...,0.007569,0.773154,0.005475,0.784653,0.005101,0.777313,0.005193,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3290...",0.679415,0.690245
50,adult,cosine,7,borda,RetentionPolicy.NEVER_RETAIN,10,43957.8,4884.2,0.110442,0.001545,...,0.008190,0.775737,0.005715,0.790344,0.004879,0.779878,0.005368,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3343...",0.678879,0.690597
67,adult,cosine,5,borda,RetentionPolicy.DD_RETENTION,10,43957.8,4884.2,0.061396,0.001638,...,0.007620,0.772993,0.005426,0.784673,0.004934,0.777186,0.005129,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3292...",0.679028,0.689930
58,adult,cosine,5,borda,RetentionPolicy.DIFFERENT_CLASS_RETENTION,10,43957.8,4884.2,0.090656,0.023909,...,0.007834,0.772629,0.005736,0.783854,0.005451,0.776761,0.005477,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3285...",0.678757,0.689966
49,adult,cosine,5,borda,RetentionPolicy.NEVER_RETAIN,10,43957.8,4884.2,0.109415,0.006005,...,0.007160,0.772330,0.005313,0.783875,0.005244,0.776514,0.005101,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3288...",0.678583,0.688826
4,adult,cosine,5,modified_plurality,RetentionPolicy.ALWAYS_RETAIN,10,43957.8,4884.2,0.110233,0.003656,...,0.006504,0.776986,0.004807,0.793006,0.004355,0.780585,0.004433,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3372...",0.678882,0.688186
22,adult,cosine,5,modified_plurality,RetentionPolicy.DIFFERENT_CLASS_RETENTION,10,43957.8,4884.2,0.110690,0.002301,...,0.006471,0.776336,0.004592,0.792146,0.004066,0.780122,0.004305,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3365...",0.678702,0.687961


In [18]:
lb_col = f"{PRIMARY_METRIC}_ci_lo"

# Sort by strongest conservative performance (higher CI lower bound first)
topN_by_lb = (
    survivors
    .sort_values(lb_col, ascending=False)
    .head(TOP_N)
    .copy()
)

# Save the configs for rerun
minimal_cols = ["dataset","metric","k","vote","retention"]
rerun_df = topN_by_lb[minimal_cols]
rerun_path = OUT_DIR / RERUN_CSV
rerun_df.to_csv(rerun_path, index=False)

topN_by_lb

,dataset,metric,k,vote,retention,num_folds,n_train_mean,n_test_mean,fit_time_s_mean,fit_time_s_std,...,f1_macro_std,precision_weighted_mean,precision_weighted_std,recall_weighted_mean,recall_weighted_std,f1_weighted_mean,f1_weighted_std,confusion_matrix_json,f1_macro_mean_ci_lo,f1_macro_mean_ci_hi
41,adult,cosine,7,borda,RetentionPolicy.ALWAYS_RETAIN,10,43957.8,4884.2,0.109486,0.004580,...,0.007056,0.776686,0.004927,0.791184,0.004314,0.780747,0.004624,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3345...",0.680923,0.691018
68,adult,cosine,7,borda,RetentionPolicy.DD_RETENTION,10,43957.8,4884.2,0.062214,0.001742,...,0.007323,0.776672,0.005157,0.791184,0.004551,0.780729,0.004841,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3345...",0.680687,0.691165
59,adult,cosine,7,borda,RetentionPolicy.DIFFERENT_CLASS_RETENTION,10,43957.8,4884.2,0.065031,0.001216,...,0.007338,0.776285,0.005212,0.790611,0.004631,0.780407,0.004900,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3341...",0.680515,0.691013
40,adult,cosine,5,borda,RetentionPolicy.ALWAYS_RETAIN,10,43957.8,4884.2,0.108597,0.009831,...,0.007569,0.773154,0.005475,0.784653,0.005101,0.777313,0.005193,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3290...",0.679415,0.690245
67,adult,cosine,5,borda,RetentionPolicy.DD_RETENTION,10,43957.8,4884.2,0.061396,0.001638,...,0.007620,0.772993,0.005426,0.784673,0.004934,0.777186,0.005129,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3292...",0.679028,0.689930
4,adult,cosine,5,modified_plurality,RetentionPolicy.ALWAYS_RETAIN,10,43957.8,4884.2,0.110233,0.003656,...,0.006504,0.776986,0.004807,0.793006,0.004355,0.780585,0.004433,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3372...",0.678882,0.688186
50,adult,cosine,7,borda,RetentionPolicy.NEVER_RETAIN,10,43957.8,4884.2,0.110442,0.001545,...,0.008190,0.775737,0.005715,0.790344,0.004879,0.779878,0.005368,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3343...",0.678879,0.690597
31,adult,cosine,5,modified_plurality,RetentionPolicy.DD_RETENTION,10,43957.8,4884.2,0.107083,0.004446,...,0.005133,0.776124,0.003650,0.792228,0.003403,0.779794,0.003395,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3370...",0.678758,0.686103
58,adult,cosine,5,borda,RetentionPolicy.DIFFERENT_CLASS_RETENTION,10,43957.8,4884.2,0.090656,0.023909,...,0.007834,0.772629,0.005736,0.783854,0.005451,0.776761,0.005477,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3285...",0.678757,0.689966
22,adult,cosine,5,modified_plurality,RetentionPolicy.DIFFERENT_CLASS_RETENTION,10,43957.8,4884.2,0.110690,0.002301,...,0.006471,0.776336,0.004592,0.792146,0.004066,0.780122,0.004305,"{""labels"": [""<=50K"", "">50K""], ""matrix"": [[3365...",0.678702,0.687961
